<a href="https://colab.research.google.com/github/jmgiorgi-10/LanguageActionTuning/blob/main/Action_Intent_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [6]:
from transformers import AutoModelForSequenceClassification
model = AutoModelForSequenceClassification.from_pretrained("xlm-roberta-base", num_labels=4)

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.bias    | MISSING    | 
classifier.out_proj.weight  | MISSING    | 
classifier.dense.weight     | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [7]:
import os
from google.colab import userdata
# Retrieve the secret and set it as an environment variable
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [24]:
from datasets import load_dataset, DatasetDict
raw_dataset = load_dataset("json", data_files="english_dataset.jsonl")

train_remainder = raw_dataset['train'].train_test_split(test_size=0.3, seed=42)

# 2. Split the 30% remainder equally into validation and test (15% / 15% of total)
val_test = train_remainder['test'].train_test_split(test_size=0.5, seed=42)

# 3. Combine into a clean 3-way DatasetDict
dataset = DatasetDict({
    'train': train_remainder['train'],       # 700 rows
    'validation': val_test['train'],         # 150 rows -> pass to Trainer(eval_dataset=...)
    'test': val_test['test']                 # 150 rows -> keep completely hidden for final eval
})


# Inspect the structure
print(dataset)
print("\nFirst training sample:")
print(dataset["train"][0])
print(dataset["test"][0])

Generating train split: 0 examples [00:00, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['user_query', 'intent'],
        num_rows: 3500
    })
    validation: Dataset({
        features: ['user_query', 'intent'],
        num_rows: 750
    })
    test: Dataset({
        features: ['user_query', 'intent'],
        num_rows: 750
    })
})

First training sample:
{'user_query': 'What are the swell conditions?', 'intent': 'check_surf'}
{'user_query': "Message Dave that I'm outside", 'intent': 'send_message'}


In [25]:
## compute_metrics just for action intent classification

import evaluate
import numpy as np

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_metrics(eval_pred):
    intent_logits, intent_labels = eval_pred
    intent_preds = np.argmax(intent_logits, axis=-1)

    # 1. Calculate accuracy
    acc = accuracy_score(intent_labels, intent_preds)

    # 2. Calculate Precision, Recall, and F1 all at once
    precision, recall, f1, _ = precision_recall_fscore_support(
        intent_labels,
        intent_preds,
        average="macro",
        zero_division=0
    )

    # 3. Return the clean dictionary
    return {
        "accuracy": acc,
        "precision_macro": precision,
        "recall_macro": recall,
        "f1_macro": f1
    }

In [26]:
# Tokenize the dataset

from transformers import AutoTokenizer

model_checkpoint = "xlm-roberta-base" # RoBERTa Fine-tuned on multi-language corpus.
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

# Define a standard ChatML template if it's missing
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)

special_tokens_dict = {'additional_special_tokens': ['<|im_start|>', '<|im_end|>']}
num_added_toks = tokenizer.add_special_tokens(special_tokens_dict)

# Crucial: Resize the model embedding layer to match the new token count!
model.resize_token_embeddings(len(tokenizer))

# Extract unique intents dynamically from your dataset
unique_intents = sorted(dataset['train'].unique("intent"))

print(dataset)

# Create lookup dictionaries
intent2id = {intent: idx for idx, intent in enumerate(unique_intents)}
id2intent = {idx: intent for idx, intent in enumerate(unique_intents)}

def preprocess_function(examples):
    prompts = examples["user_query"]
    # Map the string intents to integers using our dictionary
    intent_labels = [intent2id[intent] for intent in examples["intent"]]

    formatted_texts = []
    for prompt in prompts:
        messages = [
            {"role": "user", "content": prompt}
        ]
        # Format with the model's official template
        text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        formatted_texts.append(text)

    # Tokenize the user prompts
    tokenized = tokenizer(
        formatted_texts,
        truncation=True,
        max_length=512,
        padding=False,
    )

    # Assign the integer IDs to labels
    tokenized["labels"] = intent_labels

    return tokenized

tokenized_dataset = dataset.map(preprocess_function, batched=True)

print(tokenized_dataset)

Flattening the indices:   0%|          | 0/3500 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['user_query', 'intent'],
        num_rows: 3500
    })
    validation: Dataset({
        features: ['user_query', 'intent'],
        num_rows: 750
    })
    test: Dataset({
        features: ['user_query', 'intent'],
        num_rows: 750
    })
})


Map:   0%|          | 0/3500 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

Map:   0%|          | 0/750 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['user_query', 'intent', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3500
    })
    validation: Dataset({
        features: ['user_query', 'intent', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 750
    })
    test: Dataset({
        features: ['user_query', 'intent', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 750
    })
})


In [27]:
from peft import LoraConfig, get_peft_model, TaskType

peft_config = LoraConfig(
    r=1,
    lora_alpha=1,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
    modules_to_save=["classifier"]
)

model = get_peft_model(model, peft_config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [12]:
model

PeftModelForSequenceClassification(
  (base_model): LoraModel(
    (model): XLMRobertaForSequenceClassification(
      (classifier): ModulesToSaveWrapper(
        (original_module): XLMRobertaClassificationHead(
          (dense): Linear(in_features=768, out_features=768, bias=True)
          (dropout): Dropout(p=0.1, inplace=False)
          (out_proj): Linear(in_features=768, out_features=4, bias=True)
        )
        (modules_to_save): ModuleDict(
          (default): XLMRobertaClassificationHead(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
            (out_proj): Linear(in_features=768, out_features=4, bias=True)
          )
        )
      )
      (roberta): XLMRobertaModel(
        (embeddings): XLMRobertaEmbeddings(
          (word_embeddings): Embedding(250004, 768, padding_idx=1)
          (token_type_embeddings): Embedding(1, 768)
          (LayerNorm): LayerNorm((768,), eps=1e-05, elementwis

In [29]:
from transformers import Trainer, TrainingArguments, DataCollatorWithPadding


print(tokenized_dataset)


data_collator = DataCollatorWithPadding(tokenizer=tokenizer, return_tensors="pt")

training_arguments = TrainingArguments(
  output_dir="./results",
  num_train_epochs=15, # Let it look at the data 5 times
  logging_steps=1, # Log training loss every single step (since dataset is small)
  eval_strategy="epoch", # Calculate validation loss after each epoch
  save_strategy="epoch", # Save a checkpoint after each epoch
  learning_rate=8e-5,
  warmup_ratio = 0.1,
  per_device_train_batch_size=8,
  per_device_eval_batch_size=8,
  gradient_accumulation_steps=1,
  weight_decay=0.01,
  load_best_model_at_end=True, # Keeps the best version of the model when done
  metric_for_best_model="eval_loss"
)


trainer = Trainer(
  model=model, # The adapter-wrapped sequence classification model
  args=training_arguments,
  train_dataset = tokenized_dataset['train'],
  eval_dataset = tokenized_dataset['test'],
  data_collator = data_collator
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


DatasetDict({
    train: Dataset({
        features: ['user_query', 'intent', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 3500
    })
    validation: Dataset({
        features: ['user_query', 'intent', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 750
    })
    test: Dataset({
        features: ['user_query', 'intent', 'input_ids', 'attention_mask', 'labels'],
        num_rows: 750
    })
})


In [30]:
## It looks like it's randomly guessing between the four categories.
#### --> training data (~5.000) in just 1 language first (English US)

trainer.train()


Epoch,Training Loss,Validation Loss
1,0.235307,0.027516
2,0.000593,0.000699
3,0.000005,0.000525


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


KeyboardInterrupt: 

In [31]:
final_test_results = trainer.predict(tokenized_dataset['test'])

# Extract predictions to print your classification report
preds = np.argmax(final_test_results.predictions, axis=-1)
labels = final_test_results.label_ids

Epoch,Training Loss,Validation Loss
1,0.235307,0.027516
2,0.000593,0.000699
3,0.000005,0.000525


In [1]:
from sklearn.metrics import classification_report, accuracy_score, f1_score

# Print the comprehensive report
print("--- Classification Report ---")
print(classification_report(labels, preds, target_names=unique_intents))

# Alternatively, grab the overall metrics as isolated variables
overall_accuracy = accuracy_score(labels, preds)
overall_f1 = f1_score(labels, preds, average='weighted') # Use 'macro' or 'binary' depending on your task

print(f"Overall Accuracy: {overall_accuracy:.4f}")
print(f"Weighted F1-Score: {overall_f1:.4f}")

--- Classification Report ---


NameError: name 'labels' is not defined